# Ablation Studies

## Setup

In [ ]:
import copy
import os
import pickle
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)

from config import BATCH_SIZE, SEED
from src.dataloader import get_dataloaders
from src.losses import entropy_calibrated_kl_loss, js_divergence_loss, kl_divergence_loss
from src.model import DissagreementPredictor
from src.utils import set_seed

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FIGURES_DIR = PROJECT_ROOT / "figures"
CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DEVICE

In [ ]:
def load_cifar_batch(batch_path):
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message=r"dtype\(\): align should be passed as Python or NumPy boolean.*",
            category=np.exceptions.VisibleDeprecationWarning,
        )
        with open(batch_path, "rb") as f:
            batch = pickle.load(f, encoding="bytes")
    return batch


def resolve_checkpoint_path(model_path):
    model_path = Path(model_path)
    if model_path.exists():
        return model_path

    candidate_names = [
        model_path.name,
        model_path.name.replace("best_model_", "best_model _"),
        model_path.name.replace(" ", ""),
    ]

    for candidate_name in candidate_names:
        candidate_path = model_path.with_name(candidate_name)
        if candidate_path.exists():
            return candidate_path

    matches = sorted(model_path.parent.glob(f"*{model_path.stem.split('_')[-1]}*.pt"))
    if len(matches) == 1:
        return matches[0]

    raise FileNotFoundError(
        f"Could not find checkpoint for {model_path.name}. Available files: "
        f"{sorted(path.name for path in model_path.parent.glob('*'))}"
    )


batch = load_cifar_batch(PROJECT_ROOT / "data/raw/cifar-10-batches-py/test_batch")
images = batch[b"data"]
probs = np.load(PROJECT_ROOT / "data/raw/cifar10h-probs.npy")
train_loader, val_loader, test_loader = get_dataloaders(images, probs, batch_size=BATCH_SIZE)

len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset)

In [ ]:
def compute_distribution_metrics(all_preds, all_true, eps=1e-8):
    preds = np.clip(all_preds, eps, 1.0)
    true = np.clip(all_true, eps, 1.0)

    kl_values = np.sum(true * (np.log(true) - np.log(preds)), axis=1)
    m = 0.5 * (true + preds)
    jsd_values = 0.5 * np.sum(true * (np.log(true) - np.log(m)), axis=1)
    jsd_values += 0.5 * np.sum(preds * (np.log(preds) - np.log(m)), axis=1)
    cosine_values = np.sum(true * preds, axis=1) / (
        np.linalg.norm(true, axis=1) * np.linalg.norm(preds, axis=1) + eps
    )

    return {
        "kl_mean": float(np.mean(kl_values)),
        "jsd_mean": float(np.mean(jsd_values)),
        "cosine_mean": float(np.mean(cosine_values)),
    }


def collect_predictions(model, data_loader):
    all_preds = []
    all_true = []
    model.eval()
    with torch.no_grad():
        for imgs, targets in data_loader:
            imgs = imgs.to(DEVICE)
            preds = model(imgs)
            all_preds.append(preds.cpu().numpy())
            all_true.append(targets.numpy())
    all_preds = np.concatenate(all_preds, axis=0)
    all_true = np.concatenate(all_true, axis=0)
    return all_preds, all_true


def evaluate_loader(model, data_loader):
    preds, true = collect_predictions(model, data_loader)
    return compute_distribution_metrics(preds, true)


def freeze_backbone(model):
    for param in model.backbone.parameters():
        param.requires_grad = False


def freeze_early_backbone_layers(model):
    for idx, module in enumerate(model.backbone):
        if idx <= 4:
            for param in module.parameters():
                param.requires_grad = False


def build_model(config):
    pretrained_arg = "imagenet" if config.get("init") == "imagenet" else None
    model = DissagreementPredictor(head=config.get("head", "linear"), pretrained=pretrained_arg).to(DEVICE)

    if config.get("init") == "cifar10":
        backbone_path = resolve_checkpoint_path(PROJECT_ROOT / config["backbone_path"])
        state_dict = torch.load(backbone_path, map_location=DEVICE)
        model.backbone.load_state_dict(state_dict)

    if config.get("freeze_backbone", False):
        freeze_backbone(model)

    if config.get("freeze_early_layers", False):
        freeze_early_backbone_layers(model)

    return model


def get_loss_fn(loss_name):
    if loss_name == "kl":
        return kl_divergence_loss
    if loss_name == "jsd":
        return js_divergence_loss
    if loss_name == "custom":
        return entropy_calibrated_kl_loss
    raise ValueError(f"Unsupported loss: {loss_name}")


def train_once(model, train_loader, val_loader, config):
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(params, lr=config["lr"], weight_decay=config.get("weight_decay", 1e-4))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["epochs"])
    loss_fn = get_loss_fn(config["loss"])

    best_state = copy.deepcopy(model.state_dict())
    best_val_jsd = float("inf")
    history = []

    for epoch in range(config["epochs"]):
        model.train()
        train_losses = []

        for imgs, targets in train_loader:
            imgs = imgs.to(DEVICE)
            targets = targets.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = loss_fn(outputs, targets)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        scheduler.step()
        val_metrics = evaluate_loader(model, val_loader)
        history.append({
            "epoch": epoch + 1,
            "train_loss": float(np.mean(train_losses)),
            "val_jsd": val_metrics["jsd_mean"],
            "val_kl": val_metrics["kl_mean"],
        })

        if val_metrics["jsd_mean"] < best_val_jsd:
            best_val_jsd = val_metrics["jsd_mean"]
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"{config['name']} | epoch {epoch + 1}/{config['epochs']} | "
            f"train_loss={history[-1]['train_loss']:.4f} | val_jsd={val_metrics['jsd_mean']:.4f}"
        )

    model.load_state_dict(best_state)
    return history, best_state


def run_experiment(config):
    set_seed(SEED)
    model = build_model(config)
    history, best_state = train_once(model, train_loader, val_loader, config)
    model.load_state_dict(best_state)

    val_metrics = evaluate_loader(model, val_loader)
    test_metrics = evaluate_loader(model, test_loader)

    return {
        "config": config,
        "history": history,
        "val_jsd": val_metrics["jsd_mean"],
        "val_kl": val_metrics["kl_mean"],
        "test_jsd": test_metrics["jsd_mean"],
        "test_kl": test_metrics["kl_mean"],
        "test_cosine": test_metrics["cosine_mean"],
    }


def evaluate_existing_checkpoint(model_path, config):
    set_seed(SEED)
    model = build_model(config)
    state_dict = torch.load(resolve_checkpoint_path(model_path), map_location=DEVICE)
    model.load_state_dict(state_dict)
    val_metrics = evaluate_loader(model, val_loader)
    test_metrics = evaluate_loader(model, test_loader)
    return {
        "config": config,
        "val_jsd": val_metrics["jsd_mean"],
        "val_kl": val_metrics["kl_mean"],
        "test_jsd": test_metrics["jsd_mean"],
        "test_kl": test_metrics["kl_mean"],
        "test_cosine": test_metrics["cosine_mean"],
    }


def results_to_frame(results_dict, label_name):
    rows = []
    for label, result in results_dict.items():
        rows.append({
            label_name: label,
            "val_jsd": result["val_jsd"],
            "test_kl": result["test_kl"],
            "test_jsd": result["test_jsd"],
            "test_cosine": result["test_cosine"],
        })
    return pd.DataFrame(rows)


def plot_grouped_bars(df, category_col, title, save_path):
    x = np.arange(len(df))
    width = 0.35

    plt.figure(figsize=(8, 4.5))
    plt.bar(x - width / 2, df["val_jsd"], width, label="Val JSD")
    plt.bar(x + width / 2, df["test_kl"], width, label="Test KL")
    plt.xticks(x, df[category_col])
    plt.ylabel("Metric")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

## Ablation A — Initialization

In [ ]:
ablation_results = {}

init_experiments = [
    {
        "name": "Random",
        "init": "random",
        "head": "linear",
        "loss": "kl",
        "epochs": 40,
        "lr": 1e-3,
    },
    {
        "name": "ImageNet",
        "init": "imagenet",
        "head": "linear",
        "loss": "kl",
        "epochs": 40,
        "lr": 1e-3,
    },
    {
        "name": "CIFAR10",
        "init": "cifar10",
        "backbone_path": "checkpoints/backbone_cifar10_pretrained.pt",
        "head": "linear",
        "loss": "kl",
        "epochs": 40,
        "lr": 1e-3,
    },
]

ablation_results["initialization"] = {}
for config in init_experiments:
    ablation_results["initialization"][config["name"]] = run_experiment(config)

init_df = results_to_frame(ablation_results["initialization"], "Initialization")
init_df

In [ ]:
plot_grouped_bars(
    init_df,
    "Initialization",
    "Ablation A: Initialization",
    FIGURES_DIR / "ablation_init.png",
)

## Ablation B — Loss Functions

In [ ]:
loss_experiments = {
    "KL": {
        "checkpoint": CHECKPOINTS_DIR / "best_model_kl.pt",
        "config": {"name": "KL", "init": "cifar10", "backbone_path": "checkpoints/backbone_cifar10_pretrained.pt", "head": "linear"},
    },
    "JSD": {
        "checkpoint": CHECKPOINTS_DIR / "best_model_jsd.pt",
        "config": {"name": "JSD", "init": "cifar10", "backbone_path": "checkpoints/backbone_cifar10_pretrained.pt", "head": "linear"},
    },
    "Custom": {
        "checkpoint": CHECKPOINTS_DIR / "best_model_custom.pt",
        "config": {"name": "Custom", "init": "cifar10", "backbone_path": "checkpoints/backbone_cifar10_pretrained.pt", "head": "linear"},
    },
}

ablation_results["loss"] = {}
for label, spec in loss_experiments.items():
    ablation_results["loss"][label] = evaluate_existing_checkpoint(spec["checkpoint"], spec["config"])

loss_df = results_to_frame(ablation_results["loss"], "Loss")
loss_df

In [ ]:
plot_grouped_bars(
    loss_df,
    "Loss",
    "Ablation B: Loss Functions",
    FIGURES_DIR / "ablation_loss.png",
)

## Ablation C — Training Strategy

In [ ]:
strategy_experiments = [
    {
        "name": "SoftOnly",
        "init": "random",
        "head": "linear",
        "loss": "kl",
        "epochs": 10,
        "lr": 1e-4,
    },
    {
        "name": "Pretrain+FineTune",
        "init": "cifar10",
        "backbone_path": "checkpoints/backbone_cifar10_pretrained.pt",
        "head": "linear",
        "loss": "kl",
        "epochs": 10,
        "lr": 1e-4,
        "freeze_early_layers": True
    }
]

ablation_results["strategy"] = {}
for config in strategy_experiments:
    ablation_results["strategy"][config["name"]] = run_experiment(config)

strategy_df = results_to_frame(ablation_results["strategy"], "Strategy")
strategy_df

In [ ]:
plot_grouped_bars(
    strategy_df,
    "Strategy",
    "Ablation C: Training Strategy",
    FIGURES_DIR / "ablation_strategy.png",
)

## Ablation D — Prediction Head

In [ ]:
head_experiments = [
    {
        "name": "Linear",
        "init": "cifar10",
        "backbone_path": "checkpoints/backbone_cifar10_pretrained.pt",
        "head": "linear",
        "loss": "kl",
        "epochs": 30,
        "lr": 1e-3,
        "freeze_backbone": True
    },
    {
        "name": "MLP",
        "init": "cifar10",
        "backbone_path": "checkpoints/backbone_cifar10_pretrained.pt",
        "head": "mlp",
        "loss": "kl",
        "epochs": 30,
        "lr": 1e-3,
        "freeze_backbone": True
    },
    {
        "name": "Temperature",
        "init": "cifar10",
        "backbone_path": "checkpoints/backbone_cifar10_pretrained.pt",
        "head": "temperature",
        "loss": "kl",
        "epochs": 30,
        "lr": 1e-3,
        "freeze_backbone": True
    },
]

ablation_results["head"] = {}
for config in head_experiments:
    ablation_results["head"][config["name"]] = run_experiment(config)

head_df = results_to_frame(ablation_results["head"], "Head")
head_df

In [ ]:
plot_grouped_bars(
    head_df,
    "Head",
    "Ablation D: Prediction Head",
    FIGURES_DIR / "ablation_head.png",
)

## Final Results Tables

In [ ]:
display(init_df)
display(loss_df)
display(strategy_df)
display(head_df)